In [1]:
import pandas as pd
import numpy as np
import os
from glob import glob
from sklearn.metrics import roc_curve, auc, balanced_accuracy_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from scipy.optimize import brentq
from scipy.interpolate import interp1d
import itertools


In [2]:
def plot_confusion_matrix(labels, preds, classes=("Real", "Fake"),
                          normalize=True,
                          cmap=plt.cm.Blues):
    
    # Compute confusion matrix
    cm = confusion_matrix(labels, preds)

    # Normalize BEFORE plotting
    if normalize:
        cm = cm.astype('float') / np.maximum(cm.sum(axis=1)[:, np.newaxis], 1)

    # Plot
    plt.figure(figsize=(3, 3))
    plt.imshow(cm, interpolation='nearest', cmap=cmap,
               vmin=0, vmax=1 if normalize else None)

    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    # Annotation
    threshold = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            text = f"{cm[i, j]*100:.2f}%"
        else:
            text = f"{cm[i, j]:d}"
        
        plt.text(j, i, text,
                 horizontalalignment="center",
                 color="white" if cm[i, j] > threshold else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    
    return cm


def print_metrics(scores, labels, threshold):
    
    def compute_cllr(scores, labels):
        scores = np.array(scores)
        labels = np.array(labels)
        tar = scores[labels == 1]
        non = scores[labels == 0]
        c1 = np.mean(np.logaddexp(0, -tar) / np.log(2))
        c2 = np.mean(np.logaddexp(0, non) / np.log(2))
        return (c1 + c2) / 2
    
    fpr, tpr, thres = roc_curve(labels, scores)
    rocauc = auc(fpr, tpr)
    fnr = 1 - tpr
    eer_func = interp1d(fpr, fnr)
    eer = brentq(lambda x: eer_func(x) - x, 0., 1.)
    cllr = compute_cllr(scores, labels)
    
    preds = (scores >= threshold).astype(int)
    
    bacc = balanced_accuracy_score(preds, labels)
    acc = accuracy_score(preds, labels)
    prec = precision_score(labels, preds)
    rec = recall_score(labels, preds)
    f1 = f1_score(labels, preds)

    return rocauc, eer, cllr, bacc, acc, prec, rec, f1

def analyze_ensemble(all_logLR, labels, threshold=0):
    rocauc, eer, cllr, bacc, acc, prec, rec, f1 = print_metrics(all_logLR, labels, threshold)
    
    header = f"{'AUC%':>8} {'EER%':>8} {'Cllr':>8} | {'B.Acc.':>8} {'Acc.':>8} {'Prec.':>8} {'Recall':>8} {'F1':>8}"
    print(header)
    print(f"{rocauc*100:>8.2f} "
          f"{eer*100:>8.2f} "
          f"{cllr:>8.4f} |"
          f"{bacc*100:>8.2f} "
          f"{acc*100:>8.2f} "
          f"{prec*100:>8.2f} "
          f"{rec*100:>8.2f} "
          f"{f1*100:>8.2f}")

    preds = (all_logLR >= threshold).astype(int)
    plot_confusion_matrix(labels, preds)


## ASVspoof 2019

In [9]:
rootpath = "/home/smg/v-viola/moe_antideepfake/results/Ensemble/asv19/experts_trained_on_mix/5_experts_FULL_EVAL"

#specific_path = 'A01-A03-A04_1000_pooled_real_from_validation'
#specific_path = 'A01-A03-A04_1000_real_from_validation'
#specific_path = 'ASVspoof2015_1000'
#specific_path = 'Pseudofakes_1000'
specific_path = 'filtering_mlp'

full_path = os.path.join(rootpath, specific_path)
print(full_path)

labels = np.load(os.path.join(full_path, "labels.npy"))
logLR_scores = np.load(os.path.join(full_path, "scores_per_expert.npy"))
attack_ids = np.load(os.path.join(full_path, "attack_ids.npy"))

'''
# Print ensemble performance metrics & confusion matrix
print()
print("Ensemble performance (PLAIN SUM):")
sum_logLR_scores = np.sum(logLR_scores, axis=0)  # sum expert contributions
analyze_ensemble(sum_logLR_scores, labels, threshold=0)
print()
'''

# Compute individual logLR
unique_attacks = np.unique(attack_ids)
mean_scores_per_attack = {}
for attack in unique_attacks:
    mask = (attack_ids == attack)
    mask = mask.squeeze()  # (1, Nsamples) -> (Nsamples)
    mean_scores_per_attack[attack] = np.mean(logLR_scores[:, mask], axis=1)

# Visualize table
df = pd.DataFrame({a: mean_scores_per_attack[a] for a in unique_attacks})
df.index = [f"Exp {i+1}" for i in range(logLR_scores.shape[0])]
df.round(2)

/home/smg/v-viola/moe_antideepfake/results/Ensemble/asv19/experts_trained_on_mix/5_experts_FULL_EVAL/filtering_mlp


,-,A01,A02,A03,A04,A07,A08,A09,A10,A11,A12
Exp 1,0.22,-0.08,0.22,0.22,-0.08,-0.09,-0.09,0.88,0.34,0.33,0.35


## ASVspoof 5

In [11]:
rootpath = "/home/smg/v-viola/moe_antideepfake/results/Ensemble/asv5/experts_trained_on_mix/5_experts_FULL_EVAL"
#specific_path = 'Cset_ASV19_pooled'
specific_path = 'filtering_mlp'

full_path = os.path.join(rootpath, specific_path)
print(full_path)

labels = np.load(os.path.join(full_path, "labels.npy"))
logLR_scores = np.load(os.path.join(full_path, "scores_per_expert.npy"))
attack_ids = np.load(os.path.join(full_path, "attack_ids.npy"))

'''
# Print ensemble performance metrics & confusion matrix
print()
print("Ensemble performance (PLAIN SUM):")
sum_logLR_scores = np.sum(logLR_scores, axis=0)  
analyze_ensemble(sum_logLR_scores, labels, threshold=-0.5)
print()
'''

# Compute individual logLR
unique_attacks = np.unique(attack_ids)
mean_scores_per_attack = {}
for attack in unique_attacks:
    mask = (attack_ids == attack)
    mask = mask.squeeze()  # (1, Nsamples) -> (Nsamples)
    mean_scores_per_attack[attack] = np.mean(logLR_scores[:, mask], axis=1)

# Visualize table
df = pd.DataFrame({a: mean_scores_per_attack[a] for a in unique_attacks})
df.index = [f"Exp {i+1}" for i in range(logLR_scores.shape[0])]
df.round(2)

/home/smg/v-viola/moe_antideepfake/results/Ensemble/asv5/experts_trained_on_mix/5_experts_FULL_EVAL/filtering_mlp


,A01,A02,A03,A04,A05,A06,A07,A08,A09,A10,A11,A12,A14,A17,A19,A21,A22,A28,A29,bonafide
Exp 1,-0.12,-0.12,-0.12,-0.11,-0.12,-0.12,-0.13,-0.07,-0.09,-0.09,-0.06,-0.05,-0.1,-0.07,-0.06,-0.08,-0.09,-0.08,-0.09,-0.06


## MLAAD

In [5]:
rootpath = "/home/smg/v-viola/moe_antideepfake/results/Ensemble/mlaad/experts_trained_on_mix/5_experts_FULL_EVAL"
specific_path = 'Cset_ASV19_pooled_FAKE'

full_path = os.path.join(rootpath, specific_path)
print(full_path)

labels = np.load(os.path.join(full_path, "labels.npy"))
logLR_scores = np.load(os.path.join(full_path, "scores_per_expert.npy"))
attack_ids = np.load(os.path.join(full_path, "attack_ids.npy"))
filenames = np.load(os.path.join(full_path, "filenames.npy"))

# Print ensemble performance metrics & confusion matrix
print()
print("Ensemble performance (PLAIN SUM):")
sum_logLR_scores = np.sum(logLR_scores, axis=0)  
#analyze_ensemble(sum_logLR_scores, labels, threshold=0)
#print()

print(logLR_scores.shape)
print(sum_logLR_scores.shape)
print()
print(labels.shape)
print(attack_ids.shape)
print(np.unique(labels))
print(np.unique(attack_ids))

# Compute individual logLR
unique_attacks = np.unique(attack_ids)
mean_scores_per_attack = {}
for attack in unique_attacks:
    mask = (attack_ids == attack)
    mask = mask.squeeze()  # (1, Nsamples) -> (Nsamples)
    mean_scores_per_attack[attack] = np.mean(logLR_scores[:, mask], axis=1)

# Visualize table
df = pd.DataFrame({a: mean_scores_per_attack[a] for a in unique_attacks})
df.index = [f"Exp {i+1}" for i in range(logLR_scores.shape[0])]
df.round(2)


/home/smg/v-viola/moe_antideepfake/results/Ensemble/mlaad/experts_trained_on_mix/5_experts_FULL_EVAL/Cset_ASV19_pooled_FAKE


FileNotFoundError: [Errno 2] No such file or directory: '/home/smg/v-viola/moe_antideepfake/results/Ensemble/mlaad/experts_trained_on_mix/5_experts_FULL_EVAL/Cset_ASV19_pooled_FAKE/labels.npy'

In [6]:
rootpath = "/home/smg/v-viola/moe_antideepfake/results/Ensemble/sc/experts_trained_on_mix/5_experts_FULL_EVAL"
specific_path = 'Cset_ASV19_pooled_REAL'

full_path = os.path.join(rootpath, specific_path)
print(full_path)

labels = np.load(os.path.join(full_path, "labels.npy"))
logLR_scores = np.load(os.path.join(full_path, "scores_per_expert.npy"))
attack_ids = np.load(os.path.join(full_path, "attack_ids.npy"))
filenames = np.load(os.path.join(full_path, "filenames.npy"))

# Print ensemble performance metrics & confusion matrix
print()
print("Ensemble performance (PLAIN SUM):")
sum_logLR_scores = np.sum(logLR_scores, axis=0)  
#analyze_ensemble(sum_logLR_scores, labels, threshold=0)
#print()


print(logLR_scores.shape)
print(sum_logLR_scores.shape)
print()
print(labels.shape)
print(attack_ids.shape)
print(np.unique(labels))
print(np.unique(attack_ids))


# Compute individual logLR
unique_attacks = np.unique(attack_ids)
mean_scores_per_attack = {}
for attack in unique_attacks:
    mask = (attack_ids == attack)
    mask = mask.squeeze()  # (1, Nsamples) -> (Nsamples)
    mean_scores_per_attack[attack] = np.mean(logLR_scores[:, mask], axis=1)

# Visualize table
df = pd.DataFrame({a: mean_scores_per_attack[a] for a in unique_attacks})
df.index = [f"Exp {i+1}" for i in range(logLR_scores.shape[0])]
df.round(2)


/home/smg/v-viola/moe_antideepfake/results/Ensemble/sc/experts_trained_on_mix/5_experts_FULL_EVAL/Cset_ASV19_pooled_REAL

Ensemble performance (PLAIN SUM):
(5, 8456)
(8456,)

(8456,)
(1, 8456)
[0]
['a00']


,a00
Exp 1,-0.01
Exp 2,-0.04
Exp 3,0.11
Exp 4,-0.47
Exp 5,-0.11
